<a href="https://colab.research.google.com/github/howonJeong/Journey_to_TinyLLAMA/blob/main/Step1_MitDeep/1.6.3_TwoLayerNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..'))
import numpy as np

In [24]:
def softmax(x):
    if x.ndim == 2:
        x = x.T
        x = x - np.max(x, axis=0)
        y = np.exp(x) / np.sum(np.exp(x), axis=0)
        return y.T

    x = x - np.max(x) # 오버플로 대책
    return np.exp(x) / np.sum(np.exp(x))

def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    # 훈련 데이터가 원-핫 벡터라면 정답 레이블의 인덱스로 반환
    if t.size == y.size:
        t = t.argmax(axis=1)

    batch_size = y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size

def numerical_gradient(f, x):
    h = 1e-4 # 0.0001
    grad = np.zeros_like(x)

    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + h
        fxh1 = f(x) # f(x+h)

        x[idx] = tmp_val - h
        fxh2 = f(x) # f(x-h)
        grad[idx] = (fxh1 - fxh2) / (2*h)

        x[idx] = tmp_val # 값 복원
        it.iternext()

    return grad

sigmoid = lambda x : 1 / (1 + np.exp(-x))

In [15]:
class simpleNet:
  def __init__(self):
    self.W = np.random.randn(4,4)
    #정규분포 랜덤 가중치로 스타토

  def predict(self, x):
    return np.dot(x, self.W)

  def loss(self, x, t):
    z = self.predict(x)
    y = softmax(z)
    loss = cross_entropy_error(y, t)

    return loss

In [16]:
net = simpleNet()
print(net.W)

[[ 0.01949139 -0.74912723 -1.99853335 -0.02805502]
 [ 0.75577166  1.00063519 -0.64976    -1.77800881]
 [-1.86785839 -0.2096597   0.17165388  0.1747449 ]
 [-0.92413076 -0.35931743  1.7932536   0.30099139]]


In [17]:
x = np.array([0.6, 0.9, 0.1, 0.1])
p = net.predict(x) #앞에 넷
print(p)

[ 0.41269042  0.39419762 -1.58741326 -1.56946731]


In [18]:
np.argmax(p)
#최댓값의 인덱스

np.int64(0)

In [19]:
t = np.array([0,0,1,0]) #정답 레이블 / 인덱스
net.loss(x, t)

np.float64(2.813150003752349)

In [20]:
def f(W):
  return net.loss(x, t)

dW = numerical_gradient(f, net.W)
print(dW)

[[ 0.26610208  0.26122633 -0.56398973  0.03666132]
 [ 0.39915312  0.3918395  -0.8459846   0.05499198]
 [ 0.04435035  0.04353772 -0.09399829  0.00611022]
 [ 0.04435035  0.04353772 -0.09399829  0.00611022]]


In [21]:
f = lambda w: net.loss(x,t)

In [22]:
print(numerical_gradient(f, net.W))
#람다 써도 되고..

[[ 0.26610208  0.26122633 -0.56398973  0.03666132]
 [ 0.39915312  0.3918395  -0.8459846   0.05499198]
 [ 0.04435035  0.04353772 -0.09399829  0.00611022]
 [ 0.04435035  0.04353772 -0.09399829  0.00611022]]


In [25]:
class TwoLayerNet:
  def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
    self.params = {}
    self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
    self.params['b1'] = np.zeros(hidden_size) #사이즈 집중
    self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
    self.params['b2'] = np.zeros(output_size)

  def predict(self, x):
    W1, W2 = self.params['W1'], self.params['W2']
    b1, b2 = self.params['b1'], self.params['b2']

    a1 = np.dot(x, W1) + b1
    z1 = sigmoid(a1)
    a2 = np.dot(z1, W2) + b2
    y = softmax(a2)

    return y

  def loss(self, x, t):
    y = self.predict(x)

    return cross_entropy_error(y, t)

  def accuracy(self, x, t):
    y = self.predict(x)
    y = np.argmax(y, axis = 1)
    t = np.argmax(t, axis = 1)

    accuracy = np.sum(y==t) / float(x.shape[0]) #왜 x.shape[0]
    return accuracy

  def numerical_gradeint(self, x, t):
    loss_W = lambda W: self.loss(x,t)

    grads = {}
    grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
    grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
    grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
    grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

    return grads